# Crown Tracking Pipeline (Multithreshold + Consensus)

This notebook runs the complete crown tracking workflow using the reusable `TreeTrackingGraph` class in `src/flask_app_tracking/tree_tracking.py`.


In [ ]:
%pip install plotly

In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

import geopandas as gpd

ROOT = Path.cwd().resolve().parents[1]
APP_DIR = ROOT / 'src' / 'flask_app_tracking'
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import tree_tracking
importlib.reload(tree_tracking)
from tree_tracking import TreeTrackingGraph

print('Imported TreeTrackingGraph from:', APP_DIR)

In [ ]:
MULTITHRESH_DIR = ROOT / 'output' / 'input_crowns_multithreshold_min0p15'
ORTHO_DIR = ROOT / 'input' / 'input_om'
OUTPUT_DIR = ROOT / 'output'

print('MULTITHRESH_DIR:', MULTITHRESH_DIR)
print('ORTHO_DIR:', ORTHO_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
tracker = TreeTrackingGraph(
    multithresh_dir=str(MULTITHRESH_DIR),
    ortho_dir=str(ORTHO_DIR),
    output_dir=str(OUTPUT_DIR),
    simplify_tol=1.0,
    resize_factor=0.1,
    max_crowns_preview=200,
)

tracker.load_multithreshold_data(
    base_threshold_tag='conf_0p45',
    load_images=True,
    align=True,
)

print('OMs:', tracker.om_ids)
print('Total crowns loaded:', sum(len(v) for v in tracker.crowns_gdfs.values()))


In [ ]:
tracker.case_configs = tracker.make_strict_aligned_configs()

tracker.build_graph_conditional(
    base_max_dist=30.0,
    overlap_gate=0.10,
    min_base_similarity=0.30,
    classify_mode='balanced',
)

quality_text, quality_metrics = tracker.quality_report()
print(quality_text)

tracker.save_text(quality_text, 'final_tracking_quality_report.txt')
tracker.save_json(quality_metrics, 'final_tracking_quality_metrics.json')


In [ ]:
hq = tracker.assemble_high_quality_chains()
clean_chains = hq['clean_chains']
extracted_backbones = hq['extracted_backbones']
all_extracted_chains = hq['all_extracted_chains']

consensus_source_chains = tracker.select_consensus_source_chains(
    hq,
    include_partial=True,
    min_partial_len=5,
    min_partial_one_to_one_ratio=0.9,
)

print('High-quality chains:', len(all_extracted_chains))
print('  Clean chains:', len(clean_chains))
print('  Extracted backbones:', len(extracted_backbones))
print('Consensus source chains:', len(consensus_source_chains))
print('  Added filtered partial chains:', len(consensus_source_chains) - len(all_extracted_chains))

In [ ]:
chains_viz_dir = OUTPUT_DIR / 'tracked_chains_visualization'
tracker.visualize_all_chains(all_extracted_chains, str(chains_viz_dir))
print('Saved chain visualizations to:', chains_viz_dir)


In [ ]:
consensus_gdf = tracker.generate_consensus_crowns(consensus_source_chains)
consensus_path = OUTPUT_DIR / 'consensus_crowns_complete_all.gpkg'
consensus_gdf.to_file(consensus_path, driver='GPKG')

summary = {
    'count': int(len(consensus_gdf)),
    'quality_counts': consensus_gdf['quality'].value_counts().to_dict(),
}
tracker.save_json(summary, 'consensus_crowns_summary.json')

print('Consensus crowns:', len(consensus_gdf))
print('Saved:', consensus_path)
print('Quality counts:', summary['quality_counts'])

In [23]:
# Export consensus crowns as OM1-aligned GeoJSON
import rasterio

# If consensus_gdf is not in memory, load the latest consensus GPKG
if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
    consensus_gpkg = OUTPUT_DIR / 'consensus_crowns_complete_all.gpkg'
    if not consensus_gpkg.exists():
        raise FileNotFoundError(f'Consensus file not found: {consensus_gpkg}')
    consensus_gdf = gpd.read_file(consensus_gpkg)

if consensus_gdf.empty:
    raise ValueError('Consensus GeoDataFrame is empty. Run consensus generation first.')

# Resolve OM1 orthomosaic path (prefer tracker state, fallback to ORTHO_DIR)
om1_ortho = None
if 'tracker' in globals() and getattr(tracker, 'file_pairs', None):
    om1_ortho = Path(tracker.file_pairs[0][1])
elif 'ORTHO_DIR' in globals():
    tifs = sorted(Path(ORTHO_DIR).glob('*.tif'))
    if tifs:
        om1_ortho = tifs[0]

if om1_ortho is None or not om1_ortho.exists():
    raise FileNotFoundError('Could not resolve OM1 orthomosaic path.')

with rasterio.open(om1_ortho) as src:
    om1_crs = src.crs

if consensus_gdf.crs is None:
    consensus_om1 = consensus_gdf.set_crs(om1_crs, allow_override=True)
else:
    consensus_om1 = consensus_gdf.to_crs(om1_crs)

# Keep valid polygons only
consensus_om1 = consensus_om1[
    consensus_om1.geometry.notnull() & ~consensus_om1.geometry.is_empty
].copy()

geojson_path = OUTPUT_DIR / 'consensus_crowns_om1_phenology.geojson'
consensus_om1.to_file(geojson_path, driver='GeoJSON')

print('OM1 orthomosaic:', om1_ortho)
print('GeoJSON exported:', geojson_path)
print('Crown count:', len(consensus_om1))
print('Quality counts:', consensus_om1['quality'].value_counts().to_dict() if 'quality' in consensus_om1.columns else 'n/a')

OM1 orthomosaic: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/input_om/sit_om1.tif
GeoJSON exported: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/consensus_crowns_om1_phenology.geojson
Crown count: 80
Quality counts: {'Clean': 51, 'Pure': 27, 'Mixed': 2}


In [ ]:
consensus_viz_dir = OUTPUT_DIR / 'consensus_crowns_visualization'
tracker.visualize_all_consensus_chains(consensus_source_chains, str(consensus_viz_dir))
print('Saved consensus visualizations to:', consensus_viz_dir)